In [11]:
from indicnlp.tokenize import sentence_tokenize, indic_tokenize
from pathlib import Path
from cclang.io.schemas import DocRaw, DocTok
from unicodedata import normalize
import re
import string


project_path = Path('/Users/ppers/scientific_work/create-complete-lang')

In [18]:
def _clean_marathi_text(text: str) -> str:
    # "static" инициализация: выполняется только при первом вызове
    if not hasattr(_clean_marathi_text, "_cfg"):
        # --- константы и шаблоны, "спрятанные" внутри функции ---
        lang_range = "\u0900-\u097F"          # диапазон деванагари
        extra_punct = "।“”‘’—–…«»"
        punctuation_range = string.punctuation + extra_punct
        digit_range = "0-9"
        whitespace_range = r"\s"

        bad_symbols_pattern = (
            f"[^{lang_range}{digit_range}{re.escape(punctuation_range)}{whitespace_range}]"
        )

        cfg = {}
        cfg["bad_symbols_re"] = re.compile(bad_symbols_pattern)
        cfg["url_re"] = re.compile(r"https?://\S+|www\.\S+")
        cfg["multispace_re"] = re.compile(r"\s+")
        cfg["multipunct_re"] = re.compile(r"([?!.,।]){2,}")

        devanagari_digits = "०१२३४५६७८९"
        ascii_digits = "0123456789"
        cfg["digit_trans"] = str.maketrans(dict(zip(devanagari_digits, ascii_digits)))

        # кэшируем всё в атрибуте функции (аналог static-полей)
        _clean_marathi_text._cfg = cfg

    cfg = _clean_marathi_text._cfg

    # --- сама очистка текста ---
    text = normalize("NFC", text)

    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")

    text = cfg["url_re"].sub(" ", text)

    text = text.translate(cfg["digit_trans"])

    text = cfg["bad_symbols_re"].sub(" ", text)

    text = cfg["multipunct_re"].sub(r"\1", text)

    text = cfg["multispace_re"].sub(" ", text).strip()

    return text

In [19]:
source_text_path = project_path / Path("data/artifacts/raw_text/45/457c49fca193fb2eafdf3931a3c5a809565cfe396bc06a7acac837bb1d39361d.jsonl")
print('reading from path: ' + str(source_text_path))
sentences = []
with open(source_text_path, 'r', encoding='utf-8') as stream:
    for page in stream:
        doc = DocRaw.model_validate_json(page)
        text = doc.text
        text = _clean_marathi_text(text)

        page_sentences = sentence_tokenize.sentence_split(text, lang='mr')
        page_sentences_tokens = map(
            lambda ss: indic_tokenize.trivial_tokenize(ss, lang='mr'), page_sentences)
        sentences.extend(page_sentences_tokens)
print(sentences[:1000])

reading from path: /Users/ppers/scientific_work/create-complete-lang/data/artifacts/raw_text/45/457c49fca193fb2eafdf3931a3c5a809565cfe396bc06a7acac837bb1d39361d.jsonl
[['॥'], ['8', ')', '9', '1', '-', '4', '4511', '-', '19', 'न', '680069', '0', '-', '4', '16', '.', ')', '2208', '-', '60', '॥'], ['4', '॥', '॥'], ['॥'], ['-', '॥'], ['॥'], ['4', '1', '-', '8', '30016', '014', '-', '4'], ['-', 'य', '(', '7', 'र', 'र', '4400192297', '20', '>', '22', '-', '<', '(', '»', '"', '5', 'र्र'], ['नरसिंह', 'चिंतामण', 'केळक', 'र्', 'यांच्या', 'गग', 'धा', '5', '-', '3', '.'], ['$', 'संपादक', 'व', 'संग्राहक', 'सदाशिव', 'विनायक', 'बापट', '.'], ['$', 'कुस्तावना', '-', 'ळेखक', 'श्री', 'ज', '.', 'म', '.', 'करंदीकर', ',', 'ची', '.'], ['ए', ',', 'एळुएलू', '-', 'बी', ',', 'विश्वस्त', ',', 'कसरी', '-', 'मराठा', 'संस्था', ',', 'पुणे', '.', 'हि', '.', 'हाक', '1870', ']', 'प्रथमावृत्ति', '[', 'सन', '1948', 'र', 'किंमत', '15', 'रुपये'], ['प्रकाशक', '!'], ['सदारिव', 'विनायक', 'डोकी', 'केसरी', 'कार्यालय', ',', '568'